# BCG X Data Science Virtual Experience — Task 1
# Business Understanding & Hypothesis Framing

**Program:** BCG X Data Science Job Simulation (Forage)  
**Client:** PowerCo — a major European gas & electricity utility  
**Date:** June 2026

---

## Context

PowerCo is a major gas and electricity utility that supplies to small and medium-sized enterprises (SMEs). The energy market has become increasingly competitive following liberalisation, and PowerCo is experiencing significant customer churn — clients switching to competitor suppliers.

PowerCo has engaged BCG to investigate **why SME customers churn** and, specifically, whether **price sensitivity is the primary driver**. If price is the main lever, a targeted discount strategy could be a cost-effective retention tool.

## 1. Problem Statement

| | |
|---|---|
| **Business problem** | Customer churn in the SME segment is eroding PowerCo's revenue base |
| **Analytical question** | Can we predict which customers will churn in the next 3 months? |
| **Hypothesis** | Churn is primarily driven by price sensitivity — customers who perceive better prices elsewhere are more likely to leave |
| **If hypothesis holds** | A 20% discount offered to high-risk customers before their contract renewal could be a viable, targeted retention strategy |
| **Success metric** | A predictive model with sufficient discriminatory power (ROC-AUC ≥ 0.75) and actionable feature insights |

## 2. Hypothesis Framing

### Main Hypothesis
> *"Churn in PowerCo's SME segment is significantly driven by the customer's sensitivity to energy prices. Customers who experience high price volatility, pay above-market rates, or have seen material price increases are more likely to churn."*

### Testable Sub-Hypotheses

| # | Sub-hypothesis | Feature proxy | Testable? |
|---|---|---|---|
| H1 | Customers paying higher off-peak energy prices churn more | `price_off_peak_var`, `forecast_price_energy_off_peak` | Yes |
| H2 | Customers with higher price volatility (variance across months) churn more | Std dev of monthly prices per customer | Yes |
| H3 | Recent price changes trigger churn | Price diff: Jan vs Dec of prior year | Yes |
| H4 | Longer-tenure customers are more loyal and churn less | `num_years_antig` | Yes |
| H5 | Customers with higher consumption are more price-sensitive (larger bills) | `cons_12m` | Yes |
| H6 | Customers with lower net margin are less profitable and more likely to churn | `net_margin`, `margin_net_pow_ele` | Yes |
| H7 | Customers with gas contracts (multi-product) churn less | `has_gas` | Yes |

## 3. Key Questions for the Subject Matter Expert (SME)

Before deep-diving into the data, we would want to align with PowerCo's SME on the following:

### Business Context
1. **What is the current annual churn rate, and how has it trended over the past 3–5 years?** (Baseline for measuring model improvement)
2. **What does a churned customer look like operationally?** Do they go to a direct competitor, or exit the market? This affects how we model the problem.
3. **Is there a contract renewal cycle?** If most churn happens at renewal, the 3-month prediction window needs to align with the renewal calendar.
4. **Has PowerCo run any previous retention campaigns?** If yes, which channels, at what cost, and with what uplift?

### Data & Definitions
5. **How is `churn` defined exactly?** Does it include partial churn (reducing volume) or only full contract termination?
6. **What does `MISSING` in `channel_sales` represent?** Is it unknown, a specific acquisition channel, or a data entry default?
7. **Are the hashed fields (id, channel_sales, origin_up) stable identifiers across time?** Can we join to historical data?
8. **What time period does the training data cover?** Knowing the observation window is critical for avoiding data leakage.

### Commercial Context
9. **What is the average customer lifetime value (CLV) for the SME segment?** This determines the maximum economically justified retention discount.
10. **What is the cost of the proposed 20% discount, and what churn reduction rate would make it break-even?**
11. **Are there regulatory constraints on pricing or selective discounting?** Some markets restrict discriminatory pricing.

## 4. Data Sources Available

PowerCo has provided two datasets for analysis:

| Dataset | Records | Key content |
|---|---|---|
| `client_data.csv` | 14,606 customers | Customer attributes, consumption history, contract dates, margins, churn label |
| `price_data.csv` | 193,002 rows | Monthly energy prices per customer (12 months × ~14,606 customers) |

### Data Schema — client_data.csv

| Column | Type | Description |
|---|---|---|
| `id` | str | Hashed client identifier |
| `channel_sales` | str | Sales acquisition channel (hashed) |
| `cons_12m` | int | Electricity consumption — past 12 months (kWh) |
| `cons_gas_12m` | int | Gas consumption — past 12 months (kWh) |
| `cons_last_month` | int | Electricity consumption — last month (kWh) |
| `date_activ` | date | Contract activation date |
| `date_end` | date | Contract end date |
| `date_modif_prod` | date | Last product modification date |
| `date_renewal` | date | Next contract renewal date |
| `forecast_cons_12m` | float | Forecasted electricity consumption next 12m |
| `forecast_cons_year` | int | Forecasted electricity consumption next year |
| `forecast_discount_energy` | float | Forecasted energy discount value |
| `forecast_meter_rent_12m` | float | Forecasted meter rental bill next 2 months |
| `forecast_price_energy_off_peak` | float | Forecasted off-peak energy price |
| `forecast_price_energy_peak` | float | Forecasted peak energy price |
| `forecast_price_pow_off_peak` | float | Forecasted off-peak power price |
| `has_gas` | bool | Whether client also has a gas contract |
| `imp_cons` | float | Current paid consumption |
| `margin_gross_pow_ele` | float | Gross margin on power subscription |
| `margin_net_pow_ele` | float | Net margin on power subscription |
| `nb_prod_act` | int | Number of active products/services |
| `net_margin` | float | Total net margin |
| `num_years_antig` | int | Client tenure in years |
| `origin_up` | str | Electricity campaign the client first subscribed to (hashed) |
| `pow_max` | float | Subscribed power (kW) |
| `churn` | int | **Target:** 1 = churned within 3 months, 0 = retained |

### Data Schema — price_data.csv

| Column | Type | Description |
|---|---|---|
| `id` | str | Client identifier (matches client_data) |
| `price_date` | date | Month reference date |
| `price_off_peak_var` | float | Variable energy price — off-peak period |
| `price_peak_var` | float | Variable energy price — peak period |
| `price_mid_peak_var` | float | Variable energy price — mid-peak period |
| `price_off_peak_fix` | float | Fixed power price — off-peak period |
| `price_peak_fix` | float | Fixed power price — peak period |
| `price_mid_peak_fix` | float | Fixed power price — mid-peak period |

## 5. Analytical Approach

```
Task 1   →   Task 2          →   Task 3                   →   Task 4
Business     EDA &               Feature Engineering          Findings &
Understanding Data Cleaning      & Modelling                  Recommendations

• Frame        • Distributions     • Price change features   • Model performance
  hypothesis   • Missing values    • Tenure & contract       • Feature importance
• SME Qs       • Outliers            features                • Discount strategy
• Data audit   • Churn profile     • Random Forest           • Business case
               • Price patterns    • SHAP explainability     • Next steps
```

The modelling approach will be a **supervised binary classification** — predicting whether each SME customer will churn (1) or be retained (0) over the next 3 months. We will use a **Random Forest** classifier as the primary model due to its:
- Robustness to outliers and non-linear relationships
- Native feature importance ranking
- Ability to handle mixed data types without heavy preprocessing
- Interpretability via SHAP values

## 6. Expected Deliverables

| Task | Deliverable |
|---|---|
| Task 1 | This notebook — business framing, hypothesis, SME questions |
| Task 2 | EDA notebook with cleaned data exported to `data/processed/` |
| Task 3 | Modelling notebook with trained model, ROC-AUC, feature importance |
| Task 4 | Executive summary with actionable discount recommendation |

---

> **Next:** Proceed to [02_eda_data_cleaning.ipynb](02_eda_data_cleaning.ipynb) to explore and clean the data.